In [3]:
import sys
sys.path.append('../')

import numpy as np
import pickle
from pathlib import Path

def load_dataset(name):
    X = np.load(f'../data/processed/X_{name}.npy')
    with open(f'../data/processed/y_{name}.pkl', 'rb') as f:
        y = pickle.load(f)
    with open(f'../data/processed/meta_{name}.pkl', 'rb') as f:
        meta = pickle.load(f)
    return X, y, meta

X_vbl,    y_vbl,    meta_vbl    = load_dataset('vbl')
X_cwru,   y_cwru,   meta_cwru   = load_dataset('cwru')
X_cmapss, y_cmapss, meta_cmapss = load_dataset('cmapss')
X_mf,     y_mf,     meta_mf     = load_dataset('mf')

# Extraire uniquement les données saines de chaque dataset
def get_normal(X, y, normal_label):
    idx = [i for i, lbl in enumerate(y) if lbl == normal_label]
    return X[idx]

X_normal_vbl   = get_normal(X_vbl,   y_vbl, 'normal')   # VBL utilise 'normal'
X_normal_cwru  = get_normal(X_cwru,  y_cwru, 'sain')    # CWRU utilise 'sain'
X_normal_mf    = get_normal(X_mf,    y_mf,   'sain')    # Mechanical Faults utilise 'sain'

# CMAPSS : état sain = RUL > 80
rul_values     = np.array([m['rul'] for m in meta_cmapss])
X_normal_cmapss = X_cmapss[rul_values > 80]

print("Données saines collectées :")
print(f"  VBL-VA001        : {X_normal_vbl.shape}")
print(f"  CWRU             : {X_normal_cwru.shape}")
print(f"  Mechanical Faults: {X_normal_mf.shape}")
print(f"  CMAPSS (RUL>80)  : {X_normal_cmapss.shape}")

Données saines collectées :
  VBL-VA001        : (100, 45)
  CWRU             : (3531, 15)
  Mechanical Faults: (9000, 132)
  CMAPSS (RUL>80)  : (9631, 126)


In [4]:
from backend.ml.anomaly.autoencoder import AnomalyDetector
from pathlib import Path
import numpy as np

Path('../models').mkdir(exist_ok=True)

detectors = {}

# Nettoyage et clipping pour CMAPSS (pour éviter les NaN)
def clean_data(X):
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    # Limiter aux valeurs extrêmes (10 écarts-types) pour stabiliser l'entraînement
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    lower = mean - 10 * std
    upper = mean + 10 * std
    return np.clip(X, lower, upper)

# Boucle pour les trois premiers datasets (pas de problème)
for name, X_normal, X_all, y_all in [('VBL', X_normal_vbl, X_vbl, y_vbl),
                                      ('CWRU', X_normal_cwru, X_cwru, y_cwru),
                                      ('MF', X_normal_mf, X_mf, y_mf)]:
    print(f"\n{'='*50}")
    print(f"Entraînement Autoencoder — {name}")
    print(f"  Données saines : {X_normal.shape}")
    print(f"  Input dim      : {X_normal.shape[1]}")
    detector = AnomalyDetector(input_dim=X_normal.shape[1], latent_dim=16)
    detector.fit(X_normal, epochs=100, batch_size=256, lr=1e-3, threshold_percentile=95.0, verbose=True)
    detectors[name] = detector
    detector.save(f'../models/autoencoder_{name.lower()}.pt')

# Traitement spécial pour CMAPSS
print(f"\n{'='*50}")
print(f"Entraînement Autoencoder — CMAPSS (avec prétraitement)")
X_normal_cmapss_clean = clean_data(X_normal_cmapss)
print(f"  Données saines : {X_normal_cmapss_clean.shape}")
print(f"  Input dim      : {X_normal_cmapss_clean.shape[1]}")
detector_cmapss = AnomalyDetector(input_dim=X_normal_cmapss_clean.shape[1], latent_dim=16)
detector_cmapss.fit(
    X_normal_cmapss_clean,
    epochs=100,
    batch_size=256,
    lr=1e-4,                     # learning rate plus faible
    threshold_percentile=95.0,
    verbose=True
)
detectors['CMAPSS'] = detector_cmapss
detector_cmapss.save('../models/autoencoder_cmapss.pt')


Entraînement Autoencoder — VBL
  Données saines : (100, 45)
  Input dim      : 45
  Epoch  10/100 | Loss: 0.497620
  Epoch  20/100 | Loss: 0.263844
  Epoch  30/100 | Loss: 0.212981
  Epoch  40/100 | Loss: 0.171009
  Epoch  50/100 | Loss: 0.141823
  Epoch  60/100 | Loss: 0.134730
  Epoch  70/100 | Loss: 0.132201
  Epoch  80/100 | Loss: 0.116013
  Epoch  90/100 | Loss: 0.104853
  Epoch 100/100 | Loss: 0.107103

  Seuil anomalie (P95) : 0.104932
  Erreur min : 0.011475
  Erreur max : 0.136593
  Erreur moy : 0.040688
✅ AnomalyDetector sauvegardé → ../models/autoencoder_vbl.pt

Entraînement Autoencoder — CWRU
  Données saines : (3531, 15)
  Input dim      : 15
  Epoch  10/100 | Loss: 0.134771
  Epoch  20/100 | Loss: 0.111760
  Epoch  30/100 | Loss: 0.098289
  Epoch  40/100 | Loss: 0.090458
  Epoch  50/100 | Loss: 0.086817
  Epoch  60/100 | Loss: 0.082838
  Epoch  70/100 | Loss: 0.079886
  Epoch  80/100 | Loss: 0.079809
  Epoch  90/100 | Loss: 0.078833
  Epoch 100/100 | Loss: 0.078726

  Se

In [5]:
print("=== Diagnostic des scores d'anomalie ===\n")

for name, detector in detectors.items():
    print(f"{name}:")
    if name == 'VBL':
        X_norm = X_normal_vbl
        X_fault = X_vbl[[i for i, lbl in enumerate(y_vbl) if lbl != 'normal']]
    elif name == 'CWRU':
        X_norm = X_normal_cwru
        X_fault = X_cwru[[i for i, lbl in enumerate(y_cwru) if lbl != 'sain']]
    elif name == 'MF':
        X_norm = X_normal_mf
        X_fault = X_mf[[i for i, lbl in enumerate(y_mf) if lbl != 'sain']]
    elif name == 'CMAPSS':
        X_norm = clean_data(X_normal_cmapss)
        rul_vals = np.array([m['rul'] for m in meta_cmapss])
        X_fault = clean_data(X_cmapss[rul_vals < 20])
    
    res_n = detector.predict(X_norm)
    res_f = detector.predict(X_fault)
    sn = res_n['anomaly_score']
    sf = res_f['anomaly_score']
    print(f"  Sain    : min={sn.min():.2f}, max={sn.max():.2f}, mean={sn.mean():.2f}, >50? {(sn>50).mean()*100:.1f}%")
    print(f"  Défaut  : min={sf.min():.2f}, max={sf.max():.2f}, mean={sf.mean():.2f}, >50? {(sf>50).mean()*100:.1f}%")
    print()
    

=== Diagnostic des scores d'anomalie ===

VBL:
  Sain    : min=0.00, max=100.00, mean=23.72, >50? 8.0%
  Défaut  : min=100.00, max=100.00, mean=100.00, >50? 100.0%

CWRU:
  Sain    : min=0.00, max=100.00, mean=24.42, >50? 12.0%
  Défaut  : min=100.00, max=100.00, mean=100.00, >50? 100.0%

MF:
  Sain    : min=0.00, max=100.00, mean=17.45, >50? 6.9%
  Défaut  : min=1.44, max=100.00, mean=93.41, >50? 92.8%

CMAPSS:
  Sain    : min=0.00, max=100.00, mean=31.69, >50? 19.1%
  Défaut  : min=100.00, max=100.00, mean=100.00, >50? 100.0%



In [12]:
import matplotlib.pyplot as plt
import numpy as np

# Nettoyage des scores (assure des valeurs numériques)
def clean_scores(scores):
    return np.nan_to_num(scores, nan=0.0, posinf=100.0, neginf=0.0).clip(0, 100)

# Configurations (même que précédemment)
configs = [
    ('VBL', X_normal_vbl, X_fault_vbl, 'sain', 'défaut'),
    ('CWRU', X_normal_cwru, X_fault_cwru, 'sain', 'défaut'),
    ('MF', X_normal_mf, X_fault_mf, 'sain', 'défaut'),
    ('CMAPSS', X_normal_cmapss_clean, X_fault_cmapss, 'sain (RUL>80)', 'critique (RUL<20)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, X_normal, X_fault, normal_label, fault_label) in zip(axes.flatten(), configs):
    detector = detectors[name]

    res_normal = detector.predict(X_normal)
    res_fault  = detector.predict(X_fault)

    scores_normal = clean_scores(res_normal['anomaly_score'])
    scores_fault  = clean_scores(res_fault['anomaly_score'])

    # Histogrammes avec bins automatiques et échelle linéaire
    ax.hist(scores_normal, bins='auto', alpha=0.6, color='green', label=normal_label, density=True, histtype='stepfilled')
    ax.hist(scores_fault,  bins='auto', alpha=0.6, color='red',   label=fault_label, density=True, histtype='stepfilled')
    ax.axvline(x=50, color='orange', linestyle='--', label='Seuil 50%', linewidth=2)

    # Métriques
    tp = (scores_fault > 50).mean() * 100
    tn = (scores_normal < 50).mean() * 100
    ax.set_title(f'{name}\nDétection défauts: {tp:.1f}% | Spécificité sains: {tn:.1f}%')
    ax.set_xlabel('Score d\'anomalie (%)')
    ax.set_ylabel('Densité')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 100)

plt.suptitle('Autoencoder — Séparation sain vs défaut', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/autoencoder_validation.png', dpi=150)
plt.show()

NameError: name 'X_fault_vbl' is not defined

In [7]:
# Simuler un défaut "inconnu" : signal sain + bruit impulsionnel
np.random.seed(42)
X_sain_sample = X_normal_vbl[:100]

# Ajouter un défaut impulsionnel artificiel
noise_factor = 5.0
X_inconnu    = X_sain_sample.copy()
X_inconnu   += np.random.exponential(
    scale=noise_factor,
    size=X_inconnu.shape
) * np.random.choice([-1, 1], size=X_inconnu.shape)

detector_vbl = detectors['VBL']
result_sain  = detector_vbl.predict(X_sain_sample)
result_inknu = detector_vbl.predict(X_inconnu)

print("=== Test défaut INCONNU simulé ===\n")
print(f"Score anomalie — sain    : "
      f"{result_sain['anomaly_score'].mean():.1f}% "
      f"± {result_sain['anomaly_score'].std():.1f}%")
print(f"Score anomalie — inconnu : "
      f"{result_inknu['anomaly_score'].mean():.1f}% "
      f"± {result_inknu['anomaly_score'].std():.1f}%")
print()
print(f"Taux détection défaut inconnu : "
      f"{(result_inknu['anomaly_score'] > 50).mean()*100:.1f}%")
print()
print("Message dashboard pour défaut inconnu :")
score_moyen = result_inknu['anomaly_score'].mean()
print(f"  ⚠ Anomalie détectée — Score : {score_moyen:.1f}%")
print(f"  Type de défaut : Non référencé dans la base")
print(f"  → Inspection manuelle recommandée")
print(f"  → Voulez-vous entraîner le modèle sur ce défaut ? [Oui/Non]")

=== Test défaut INCONNU simulé ===

Score anomalie — sain    : 23.7% ± 21.4%
Score anomalie — inconnu : 100.0% ± 0.0%

Taux détection défaut inconnu : 100.0%

Message dashboard pour défaut inconnu :
  ⚠ Anomalie détectée — Score : 100.0%
  Type de défaut : Non référencé dans la base
  → Inspection manuelle recommandée
  → Voulez-vous entraîner le modèle sur ce défaut ? [Oui/Non]
